### Small Language Model (Deutsch Geschiste)


In [39]:
! /opt/homebrew/bin/python3.10 -m pip install ipywidgets

1397.72s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 4.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
import json
from itertools import islice



In [ ]:
# Authentication with Hugging face
from huggingface_hub import login
import os
import ipywidgets

#load token here
login()

TOKENIZER = "dbmdz/german-gpt2"


In [ ]:
# Causal Attention Class -- Not required Anymore ---> just for understanding 
class CasualAttention(nn.Module):

    """This is the single Self Attention class"""

    def __init__(self,d_in,qvk_bias=False):
        super().__init__()
            #initailize the trainable Q,K,V Matrices
        self.trainable_Wq = nn.Linear(d_in,d_in,bias=qvk_bias)
        self.trainable_Wk = nn.Linear(d_in,d_in,bias=qvk_bias)
        self.trainable_Wv = nn.Linear(d_in,d_in,bias=qvk_bias)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self,batch_input):
        # input shape -> (batch_size,context size,embedding_dim)
        query  = self.trainable_Wq(batch_input)
        key = self.trainable_Wk(batch_input)
        value = self.trainable_Wv(batch_input)

        #Attention scores
        attention_scores = query @ key.transpose(-2, -1)

        #attention weights
        d_k = query.size(-1) 
        attention_scores = attention_scores / (d_k ** 0.5)
        #attention_weights = F.softmax(attention_scores, dim=-1) Normal Attention weights without masking
        

        #Adding Causal Attention Mask to hide the future tokens
        self.context_length = attention_scores.shape[1]
        mask = torch.triu(torch.ones(self.context_length,self.context_length),diagonal=1)
        masked_attention_scores = attention_scores.masked_fill(mask.bool(),-torch.inf)
        attention_weights = torch.softmax(masked_attention_scores/key.shape[-1]**0.5,dim=1)
        attention_weights = self.dropout(attention_weights)
        context_vector = attention_weights @ value
        print("Shape of Context Vector :",context_vector.shape)
        return context_vector

In [ ]:

class ModelConfig:
    def __init__(self):
        self.vocab_size = 50266
        self.EmbeddingVectorSize = 384
        self.ContextSize = 120
        self.EmbeddingBatchSize = 5
        self.FFSCALE = 4
        self.BLOCK_SIZE = 12
        self.NUM_HEADS = 2
        self.DROPOUT = 0.1
        #dropout = 0.1 manually set

class Tokenizer:
    """Converts Input Text into Tokens using given Tokenizer"""
    def __init__(self, tokenizer_name):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token #EOD - end of sequence 

    def encode(self, inp_batch):
        return self.tokenizer(inp_batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_tokens=120).input_ids

    def decode(self, out_batch):
        return self.tokenizer.batch_decode(out_batch)


class EmbeddingLayer(torch.nn.Module):

    """converts input Tokens into Embedding Vectors and Position Vectors """
    def __init__(self, config):
        super().__init__()

        #Embedding Matrix manually using tensor (inefficient at memory level)
                #self.embedding_matrix = nn.Parameter(torch.randn(config.vocab_size, config.EmbeddingVectorSize))

        # Embedding Matrix (optimized)
        self.embedding_matrix = nn.Embedding(num_embeddings=config.vocab_size,embedding_dim=config.EmbeddingVectorSize)
        
        self.position_matrix = nn.Embedding(num_embeddings=config.ContextSize,embedding_dim=config.EmbeddingVectorSize)

    def embedding_vector(self, input_token_matrix):
        return self.embedding_matrix(input_token_matrix)

    def position_vector(self, input_token_matrix):
        batch_size, seq_len = input_token_matrix.shape

        position_ids = torch.arange(seq_len)  # [0,1,2,...]
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)

        position_ids = position_ids.to(input_token_matrix.device)

        return self.position_matrix(position_ids)
    
    def context_vector(self,embedding_vector_matrix,position_vector_matrix):
        return embedding_vector_matrix+position_vector_matrix

    
class MultiHeadAttention(nn.Module):
    """Multi-Head Self Attention with Causal Masking"""
    def __init__(self, d_in, d_out, num_heads, qvk_bias=False, dropout=0.1):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Dimension per head

        #initalizing the mask need in causal Attention 
        self.register_buffer("mask",torch.triu(torch.ones(100, 100), diagonal=1).bool())
        
        # Initialize trainable Q, K, V matrices (project to d_out)
        self.trainable_Wq = nn.Linear(d_in, d_out, bias=qvk_bias)
        self.trainable_Wk = nn.Linear(d_in, d_out, bias=qvk_bias)
        self.trainable_Wv = nn.Linear(d_in, d_out, bias=qvk_bias)
        
        # Output projection to combine heads
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, batch_input):
        # input shape -> (batch_size, context_length, d_in)
        batch_size, context_length, d_in = batch_input.shape
        
        # Project to Q, K, V: (batch_size, context_length, d_out)
        query = self.trainable_Wq(batch_input)
        key = self.trainable_Wk(batch_input)
        value = self.trainable_Wv(batch_input)
        
        # Reshape to split into heads: (batch_size, num_heads, context_length, head_dim)
        query = query.view(batch_size, context_length, self.num_heads, self.head_dim).transpose(1, 2)
        key = key.view(batch_size, context_length, self.num_heads, self.head_dim).transpose(1, 2)
        value = value.view(batch_size, context_length, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Compute attention scores: (batch_size, num_heads, context_length, context_length)
        attention_scores = query @ key.transpose(-2, -1)
        
        # Scale by sqrt(head_dim)
        attention_scores = attention_scores / (self.head_dim ** 0.5)
        
        # Apply causal mask to hide future tokens
        mask = self.mask[:context_length, :context_length]
        mask = mask.unsqueeze(0).unsqueeze(0)
        masked_attention_scores = attention_scores.masked_fill(mask, -torch.inf)
        
        # Compute attention weights
        attention_weights = torch.softmax(masked_attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values: (batch_size, num_heads, context_length, head_dim)
        context_vector = attention_weights @ value
        
        # Concatenate heads: (batch_size, context_length, d_out)
        context_vector = context_vector.transpose(1, 2).contiguous()
        context_vector = context_vector.view(batch_size, context_length, self.d_out)
        
        # Final output projection
        output = self.out_proj(context_vector)
        
        return output
        

class FF(nn.Module):
    """  This is the Feed Forward Neural Net Class with GELU Activation"""
    def __init__(self,config):
        super().__init__()
        self.config = config
        self.layers = nn.Sequential(nn.Linear(self.config.EmbeddingVectorSize,self.config.FFSCALE *self.config.EmbeddingVectorSize),
                                    nn.GELU(),
                                    nn.Linear(self.config.FFSCALE *self.config.EmbeddingVectorSize,self.config.EmbeddingVectorSize))
        
    def forward(self,batch_context_vector):
        return self.layers(batch_context_vector)
    
class LayerNormalization(nn.Module):

    """Use premade LayerNormalizer from Pytorch"""
    def __init__(self,config):
        super().__init__()
        self.layernormalizer = nn.LayerNorm(config.EmbeddingVectorSize)

    def forward(self,inp):
        return self.layernormalizer(inp)
    

class LMHead(nn.Module):
    """Final Output Layer of the SLM Architecture"""
    def __init__(self,config):
        super().__init__()
        self.config = config 
        self.layer = nn.Linear(config.EmbeddingVectorSize, config.vocab_size) #better implementation


    def forward(self,ff_input):
        l =  self.layer(ff_input)
        return l
        # probabilities = F.softmax(l, dim=-1)
        # return probabilities
    
class Block(nn.Module):

    """ The transformer Block"""
    def __init__(self,config):
        super().__init__()
        self.config = config
        self.attention_block = MultiHeadAttention(self.config.EmbeddingVectorSize,self.config.EmbeddingVectorSize,num_heads=self.config.NUM_HEADS,qvk_bias=False,dropout=self.config.DROPOUT)
        self.layer_normalizer1 = LayerNormalization(self.config)
        self.layer_normalizer2 = LayerNormalization(self.config)
        self.dropout = nn.Dropout(config.DROPOUT)
        self.ff = FF(config=self.config)

    def forward(self,input_vector):
        normalized_inputs = self.layer_normalizer1(input_vector)
        contextVector = self.attention_block(normalized_inputs)
        #Adding dropout 
        contextVector = self.dropout(contextVector)
        #shortcut connection #1
        contextVector = contextVector+input_vector
        normalized_context_vector = self.layer_normalizer2(contextVector)
        output = self.ff(normalized_context_vector)
        #Adding Dropout
        output = self.dropout(output)
        #shortcut connection #1
        output = output+contextVector
        return output


#SLM Model 
class SLM_Model(nn.Module):

    """Setting up the SLM Architecture """

    def __init__(self):
        super().__init__()
        self.config = ModelConfig()
        self.embedding = EmbeddingLayer(config=self.config)
        self.llm_head = LMHead(self.config)
        self.layer_normalization = LayerNormalization(self.config)
        self.blocks = nn.ModuleList([Block(config=self.config) for _ in range(self.config.BLOCK_SIZE)])
        #weights tying -- > didn't understood perfectly but works 
        self.llm_head.layer.weight = self.embedding.embedding_matrix.weight
     
    def forward(self,input_batch_tokens):

        # Trim sequence lenght if larger than model context size (trim from start )
        input_batch_tokens = input_batch_tokens[:, -self.config.ContextSize:]
       
        embedding_vector = self.embedding.embedding_vector(input_batch_tokens)
        position_vector = self.embedding.position_vector(input_batch_tokens)
        input_vector = embedding_vector+position_vector
        x = input_vector
        for block in self.blocks:
            x = block(x)
        
        #final layer normalization 
        x = self.layer_normalization(x)
        print("output before normalization :",x)
        output = self.llm_head(x)
        return output

        #return torch.argmax(self.llm_head.forward(x), dim=-1)




In [301]:
#Forward pass 

#predicting the first token
tokenizer = Tokenizer(TOKENIZER)
batch_tokens = tokenizer.encode(["Es ist Sehr ","Das is super"])


In [302]:
model = SLM_Model()

output = F.softmax(model.forward(batch_tokens),dim=-1)
print("output : ",output)
torch.set_printoptions(precision=10)
print(output[0, -1, :10])

output before normalization : tensor([[[-0.8521311283, -1.2928878069,  0.6649159193,  ...,
          -0.0832689777, -0.2387481779,  0.9793520570],
         [-0.2466249019,  1.2012407780, -1.3765633106,  ...,
           0.7183226347, -1.3923866749,  0.9179574847],
         [ 0.8301240802, -0.8332967758,  0.4927342236,  ...,
           1.0558789968,  1.8573296070, -0.4016053677],
         [-0.5459007621, -0.4558581412,  0.6955782771,  ...,
           1.9240458012,  0.4754185975, -1.3333965540]],

        [[-0.1614312232,  0.1715534031,  1.0285701752,  ...,
          -0.4752151668,  1.5705000162,  0.7337675691],
         [-1.0696645975,  0.1147933602,  1.0237568617,  ...,
           0.4890799522,  0.9432908297,  0.5761982203],
         [ 0.1154332832, -0.7628180981,  0.0342425965,  ...,
           1.1712752581,  1.2235544920, -0.9254254103],
         [-0.1886610538, -0.4685214162,  0.3639445007,  ...,
           1.5750868320,  1.3670204878, -1.3460803032]]],
       grad_fn=<NativeLayerNor

<h2 style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 300;">
  <span style="background: linear-gradient(90deg, #ffffffff, #ffffffff); -webkit-background-clip: text; color: transparent;">
    LLM training
  </span>
</h2>


In [303]:
# Creating Batch of Training Data 

class Training():
    """This class creates dataset batches and then starts the training pipeline"""
    def __init__(self):
        pass

    def simple_training_batch_generator(self, file_path, start=0, end=None):
        stories = []

        with open(file_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):

                if i < start:
                    continue

                if end is not None and i >= end:
                    break

                data = json.loads(line)

                if "de" in data:
                    stories.append(data["de"])

        return stories
    
    def generate_input_target_batch(self, token_batch, context_size):
    
        # Ensure tensor
        if not isinstance(token_batch, torch.Tensor):
            token_batch = torch.tensor(token_batch, dtype=torch.long)

        # Trim to context size (keep last tokens)
        token_batch = token_batch[:, -context_size-1:]

        # Create input and target
        x = token_batch[:, :-1]
        y = token_batch[:, 1:]

        return x, y



In [306]:
tokenizer.eos_token

AttributeError: 'Tokenizer' object has no attribute 'eos_token'

In [304]:

trainer = Training()
o = trainer.simple_training_batch_generator("/Users/abhishek/German-Tiny-stories-/dataset generation/dataset.jsonl",0,3)
input,output = trainer.generate_input_target_batch(tokenizer.encode(o),1000)




In [305]:
input.shape

torch.Size([3, 103])

In [309]:
tokenizer.encode("<|endoftext|>")

tensor([[50265]])

In [310]:
tokenizer.tokenizer.eos_token_id

50265

In [ ]:
# Taking Avg Token Length as Context size

import json
import random

def calculate_token_stats(file_path, batch_size=1000):
    total_tokens = 0
    line_count = 0
    max_tokens = 0
    current_batch = []

    print(f"Starting processing: {file_path}\n")

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line)
                text = data.get("de", "")
                if text:
                    current_batch.append(text)
                
                if len(current_batch) >= batch_size:
                    # --- CRITICAL: Check if your tokenizer has a 'max_length' or 'truncation' setting ---
                    # If it does, set truncation=False to see the TRUE length of your data.
                    encoded_batch = tokenizer.encode(current_batch)
                    
                    batch_total_len = 0
                    for tokens in encoded_batch:
                        t_len = len(tokens)
                        total_tokens += t_len
                        batch_total_len += t_len
                        if t_len > max_tokens:
                            max_tokens = t_len
                    
                    line_count += len(current_batch)
                    
                    # --- REAL-TIME FEEDBACK ---
                    batch_avg = batch_total_len / len(current_batch)
                    # Pick a random sample from the current batch to verify it's not always '100'
                    sample_len = len(random.choice(encoded_batch))
                    
                    print(f"Batch {line_count // batch_size} | "
                          f"Batch Avg: {batch_avg:.1f} | "
                          f"Random Sample Length: {sample_len} | "
                          f"Running Total Avg: {total_tokens / line_count:.1f}")
                    
                    current_batch = []

            except Exception as e:
                print(f"Error: {e}")
                continue

        # Final batch
        if current_batch:
            encoded_batch = tokenizer.encode(current_batch)
            for tokens in encoded_batch:
                t_len = len(tokens)
                total_tokens += t_len
                if t_len > max_tokens:
                    max_tokens = t_len
            line_count += len(current_batch)

    print("\n" + "="*30)
    print(f"FINAL TOTAL STORIES: {line_count}")
    print(f"FINAL AVG TOKENS: {total_tokens / line_count:.2f}")
    print(f"FINAL MAX TOKENS: {max_tokens}")
    print("="*30)

file_path = "/Users/abhishek/German-Tiny-stories-/dataset generation/output (3).jsonl"
calculate_token_stats(file_path)

Starting processing: /Users/abhishek/German-Tiny-stories-/dataset generation/output (3).jsonl

Batch 1 | Batch Avg: 112.0 | Random Sample Length: 112 | Running Total Avg: 112.0
Batch 2 | Batch Avg: 115.0 | Random Sample Length: 115 | Running Total Avg: 113.5
Batch 3 | Batch Avg: 117.0 | Random Sample Length: 117 | Running Total Avg: 114.7
Batch 4 | Batch Avg: 118.0 | Random Sample Length: 118 | Running Total Avg: 115.5
Batch 5 | Batch Avg: 116.0 | Random Sample Length: 116 | Running Total Avg: 115.6
Batch 6 | Batch Avg: 116.0 | Random Sample Length: 116 | Running Total Avg: 115.7
Batch 7 | Batch Avg: 114.0 | Random Sample Length: 114 | Running Total Avg: 115.4
Batch 8 | Batch Avg: 114.0 | Random Sample Length: 114 | Running Total Avg: 115.2
Batch 9 | Batch Avg: 114.0 | Random Sample Length: 114 | Running Total Avg: 115.1
Batch 10 | Batch Avg: 115.0 | Random Sample Length: 115 | Running Total Avg: 115.1
Batch 11 | Batch Avg: 116.0 | Random Sample Length: 116 | Running Total Avg: 115.2
B

torch.Size([100, 100])

tensor([[  286, 22430, 10738,   292],
        [ 7671, 50265, 50265, 50265],
        [  555,  2587,    18,   484]])

In [260]:
output

tensor([[22430, 10738,   292,  3175],
        [50265, 50265, 50265, 50265],
        [ 2587,    18,   484, 17313]])